# E-commerce Purchase Intent & Expected Value / Ý định mua và giá trị kỳ vọng

Tôi kết hợp hai bài toán: **dự đoán xác suất mua** và **chọn hành động dựa trên expected value**.
I explore both purchase-probability prediction and simple expected-value decision rules.

**Input:** Online Shoppers Purchasing Intention dataset (OpenML/UCI), session features and historical `Revenue` label.
**Output:** predicted probability, candidate actions (no incentive / discount / retarget), EV calculations and policy comparison.
**Điều quan trọng / Key assumption:** các hàm uplift của discount/retargeting và margin/cost trong bài là **giả định mô phỏng** để học ra quyết định; tác động thực tế cần được đo riêng qua thử nghiệm phù hợp.
**Run:** cần truy cập OpenML hoặc dữ liệu CSV tương ứng; các phép tính downstream sử dụng pandas, NumPy và scikit-learn.



# Ecommerce Intent Prediction + Expected Value Decision

Notebook này xây một pipeline theo hướng **job-ready**:

**data → feature engineering → intent model → EV decision optimization → evaluation**

## Mục tiêu
1. Dự đoán xác suất khách sẽ mua (`purchase probability`)
2. Không dừng ở prediction, mà tối ưu **decision** theo expected value:

\[
EV = p(buy) \times margin - action\_cost
\]

Trong notebook này, ta so sánh:

- **No incentive**
- **Discount**
- **Retargeting**

## Dataset
Notebook ưu tiên dùng **Online Shoppers Purchasing Intention Dataset** từ OpenML/UCI.

Target:
- `Revenue` = 1 nếu session dẫn tới purchase, ngược lại = 0


In [ ]:

# If running in Colab, uncomment:
# !pip -q install openml scikit-learn pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier



## 1. Load dataset

Nếu `fetch_openml` không chạy được do mạng hoặc mirror, bạn có thể thay bằng:
- upload CSV thủ công vào Colab
- hoặc đọc từ Google Drive


In [ ]:

# Try OpenML first
try:
    # Online Shoppers Purchasing Intention Dataset
    data = fetch_openml(name="17OnlineShoppersPurchasingIntention", version=1, as_frame=True)
    df = data.frame.copy()
    print("Loaded from OpenML")
except Exception as e:
    print("OpenML load failed:", e)
    print("Please upload a CSV manually and replace this cell with pd.read_csv(...)")
    raise

print(df.shape)
df.head()


In [ ]:

df.info()



## 2. Basic cleaning

Chuẩn hóa target `Revenue` về 0/1.


In [ ]:

df = df.copy()

# Standardize target
if "Revenue" not in df.columns:
    raise ValueError("Expected target column 'Revenue' not found.")

# Convert Revenue to numeric binary
df["Revenue"] = df["Revenue"].astype(str).str.lower().map({
    "true": 1, "false": 0, "1": 1, "0": 0
}).fillna(df["Revenue"]).astype(int)

print(df["Revenue"].value_counts(dropna=False))
df.head()



## 3. Feature / target split

Ta giữ bài toán đơn giản:
- **X**: toàn bộ feature session / behavior
- **y**: `Revenue`


In [ ]:

target_col = "Revenue"
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)



## 4. Train / validation / test split

Ta chia theo random split cho demo.

Trong production thực tế, nếu có timestamp/session ordering rõ ràng,
nên dùng **time-based split** để tránh optimistic bias.


In [ ]:

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)
print("Test :", X_test.shape, y_test.shape)



## 5. Preprocessing pipeline

- Numeric: median imputation + scaling
- Categorical: most frequent imputation + one-hot encoding


In [ ]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)



## 6. Baseline model

Bắt đầu bằng Logistic Regression vì:
- dễ giải thích
- job market chấp nhận
- đủ tốt cho baseline probability model


In [ ]:

logit_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logit_model.fit(X_train, y_train)

valid_proba = logit_model.predict_proba(X_valid)[:, 1]
test_proba = logit_model.predict_proba(X_test)[:, 1]

print("Validation AUC :", roc_auc_score(y_valid, valid_proba))
print("Validation LogLoss :", log_loss(y_valid, valid_proba))
print("Validation Brier :", brier_score_loss(y_valid, valid_proba))

print("Test AUC :", roc_auc_score(y_test, test_proba))
print("Test LogLoss :", log_loss(y_test, test_proba))
print("Test Brier :", brier_score_loss(y_test, test_proba))



## 7. Optional benchmark model

Thử Random Forest như benchmark nhẹ.
Không cần model quá nặng; mục tiêu ở đây là **decision system**, không phải leaderboard model.


In [ ]:

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

valid_proba_rf = rf_model.predict_proba(X_valid)[:, 1]
test_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("RF Validation AUC :", roc_auc_score(y_valid, valid_proba_rf))
print("RF Test AUC :", roc_auc_score(y_test, test_proba_rf))



## 8. Chọn model tốt hơn

Chọn model theo validation AUC.


In [ ]:

if roc_auc_score(y_valid, valid_proba_rf) > roc_auc_score(y_valid, valid_proba):
    best_model = rf_model
    best_name = "RandomForest"
    valid_scores = valid_proba_rf
    test_scores = test_proba_rf
else:
    best_model = logit_model
    best_name = "LogisticRegression"
    valid_scores = valid_proba
    test_scores = test_proba

print("Selected model:", best_name)
print("Validation AUC:", roc_auc_score(y_valid, valid_scores))
print("Test AUC:", roc_auc_score(y_test, test_scores))



## 9. Probability diagnostics

Nhìn phân phối probability để xem model có đang phân tách tương đối hay không.


In [ ]:

plt.figure(figsize=(8,5))
plt.hist(test_scores[y_test.values == 0], bins=30, alpha=0.6, label="No Purchase")
plt.hist(test_scores[y_test.values == 1], bins=30, alpha=0.6, label="Purchase")
plt.xlabel("Predicted purchase probability")
plt.ylabel("Count")
plt.title(f"Probability Distribution - {best_name}")
plt.legend()
plt.show()



## 10. Decision optimization bằng Expected Value

Đây là phần quan trọng nhất.

Ta định nghĩa 3 action:

1. **No action**
2. **Discount**
3. **Retargeting**

### Công thức EV đơn giản

- `EV_no_action = p_buy * margin`
- `EV_discount = p_buy_discount * discounted_margin`
- `EV_retarget = p_buy_retarget * margin - retarget_cost`

Để giữ notebook rõ ràng, ta mô hình hóa uplift như giả định kinh doanh:

- discount giúp tăng xác suất mua một lượng nhỏ
- retargeting cũng tăng xác suất mua, nhưng có cost


In [ ]:

# Business assumptions (editable)
BASE_MARGIN = 50.0            # margin nếu mua, trước discount
DISCOUNT_COST = 10.0          # discount làm giảm margin
RETARGET_COST = 2.0           # cost cho retargeting

# Assumed uplift functions
def uplift_discount(p):
    # Discount giúp uplift mạnh hơn ở nhóm p trung bình
    return np.clip(0.12 * (1 - np.abs(p - 0.5) * 2), 0.02, 0.12)

def uplift_retarget(p):
    # Retargeting uplift nhẹ hơn nhưng rẻ
    return np.clip(0.05 * (1 - np.abs(p - 0.45) * 1.6), 0.01, 0.05)

def compute_expected_values(p):
    p = np.asarray(p)

    p_no = p
    ev_no = p_no * BASE_MARGIN

    p_disc = np.clip(p + uplift_discount(p), 0, 1)
    ev_discount = p_disc * (BASE_MARGIN - DISCOUNT_COST)

    p_retarget = np.clip(p + uplift_retarget(p), 0, 1)
    ev_retarget = p_retarget * BASE_MARGIN - RETARGET_COST

    return pd.DataFrame({
        "p_buy": p,
        "EV_no_action": ev_no,
        "EV_discount": ev_discount,
        "EV_retarget": ev_retarget
    })



## 11. Chọn action tối ưu theo EV


In [ ]:

ev_df = compute_expected_values(test_scores)

action_cols = ["EV_no_action", "EV_discount", "EV_retarget"]
ev_df["best_action"] = ev_df[action_cols].idxmax(axis=1)

display(ev_df.head())
ev_df["best_action"].value_counts()



## 12. So sánh với rule-based policy cũ

Rule-based:
- p > 0.7 → no action
- 0.4 < p <= 0.7 → discount
- còn lại → retarget

Policy này hợp cho demo, nhưng **EV policy** mới là mindset production hơn.


In [ ]:

def rule_based_action(p):
    if p > 0.7:
        return "EV_no_action"
    elif p > 0.4:
        return "EV_discount"
    else:
        return "EV_retarget"

rule_actions = pd.Series(test_scores).apply(rule_based_action)

# EV-optimal action
ev_actions = ev_df["best_action"]

comparison = pd.DataFrame({
    "p_buy": test_scores,
    "rule_action": rule_actions,
    "ev_action": ev_actions
})

comparison.head()



## 13. Đánh giá policy theo expected profit

Vì đây là dataset historical không có randomized treatment logs,
ta **không thể** kết luận causal uplift thật.

Do đó, notebook này dùng **simulated policy evaluation** theo business assumptions ở trên.
Điểm cộng là nó cho bạn đúng structure của một **decision system**.


In [ ]:

def expected_profit_from_action(p, action):
    if action == "EV_no_action":
        return p * BASE_MARGIN
    elif action == "EV_discount":
        p_new = min(1.0, p + float(uplift_discount(np.array([p]))[0]))
        return p_new * (BASE_MARGIN - DISCOUNT_COST)
    elif action == "EV_retarget":
        p_new = min(1.0, p + float(uplift_retarget(np.array([p]))[0]))
        return p_new * BASE_MARGIN - RETARGET_COST
    else:
        raise ValueError("Unknown action")

comparison["rule_ev"] = [
    expected_profit_from_action(p, a) for p, a in zip(comparison["p_buy"], comparison["rule_action"])
]
comparison["opt_ev"] = [
    expected_profit_from_action(p, a) for p, a in zip(comparison["p_buy"], comparison["ev_action"])
]

print("Average expected profit - Rule policy :", comparison["rule_ev"].mean())
print("Average expected profit - EV policy   :", comparison["opt_ev"].mean())
print("Expected uplift per session           :", comparison["opt_ev"].mean() - comparison["rule_ev"].mean())



## 14. Calibration buckets

Để nhìn model theo business lens:
- nhóm khách xác suất cao có mua thực sự cao hơn không?


In [ ]:

calib_df = pd.DataFrame({
    "p_buy": test_scores,
    "actual": y_test.values
}).copy()

calib_df["bucket"] = pd.qcut(calib_df["p_buy"], q=10, duplicates="drop")

bucket_summary = calib_df.groupby("bucket").agg(
    avg_pred=("p_buy", "mean"),
    actual_rate=("actual", "mean"),
    n=("actual", "size")
).reset_index()

bucket_summary


In [ ]:

plt.figure(figsize=(7,5))
plt.plot(bucket_summary["avg_pred"], bucket_summary["actual_rate"], marker="o")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("Average predicted probability")
plt.ylabel("Actual purchase rate")
plt.title("Calibration by bucket")
plt.show()



## 15. Business summary

Nếu model + policy ổn, bạn có thể viết CV theo hướng:

- Built an ecommerce intent prediction system using session-level behavioral features
- Trained and evaluated classification models for purchase probability estimation
- Designed an expected-value decision policy for discount vs retargeting actions
- Improved simulated expected profit per session compared with a rule-based baseline

## Điểm mạnh của project này
- Không dừng ở model
- Có **decision optimization**
- Có **business impact framing**
- Gần Product / Growth / Data Analyst / Decision Science



## 16. Hướng nâng cấp tiếp theo

1. **Causal uplift modeling**
   - Nếu có treatment logs thật, build uplift model thay vì assumed uplift

2. **Time-based validation**
   - Split theo thời gian thay vì random split

3. **Expected value theo SKU / category**
   - margin khác nhau theo ngành hàng

4. **Policy simulation sâu hơn**
   - budget constraint
   - discount quota
   - user fatigue
